# Phase 3: how much can the Kaggle GPU handle at each image size?

**CRISP-DM stage:** Data preparation

**Runs on:** Kaggle with the GPU and internet on, to install anomalib and fetch the pretrained networks. On the laptop it runs a tiny test only, with untrained networks, to prove the code runs.

**Which photos:** a few good Vial training photos. Vial has the most training photos (291), so it needs the most memory. No test photos are read, so half A and half B are not involved, and no scores are produced.

**What this gives:** measured time and memory on a few photos, plus estimates for a full run. Estimates are marked, and they are not results.

**PaDiM is not in this check.** Its inverse step crashed on the GPU in two earlier attempts. That step is diagnosed separately in `03_inverse_diagnosis.ipynb`, and PaDiM's options are decided on their own. WinCLIP is not here either, because anomalib fixes its size at 240 x 240.

In [ ]:
!pip install anomalib==2.7.0

## 1. Settings

- **Sizes:** 256 is anomalib's default, 1024 is 4 times larger each way, and the sizes between show where costs rise. Memory for a square photo depends on its total pixels, so these numbers also apply if a photo is padded to a square.
- **8 photos per batch:** a batch is the group of photos processed together. 8 is a working choice, small enough to test quickly. A larger batch uses more memory, so the batch size used later must be checked against these numbers.
- **291 photos:** the number of Vial training photos, used only for estimates.
- **Network:** Wide ResNet-50 is the largest network in this project's scope, so it shows the worst case.

In [ ]:
import csv
import gc
import time
from pathlib import Path

import torch
from anomalib.data.utils import read_image
from anomalib.models import Padim
from anomalib.models.image.patchcore.torch_model import PatchcoreModel
from anomalib.models.image.reverse_distillation.anomaly_map import AnomalyMapGenerationMode
from anomalib.models.image.reverse_distillation.loss import ReverseDistillationLoss
from anomalib.models.image.reverse_distillation.torch_model import ReverseDistillationModel

USING_GPU = torch.cuda.is_available()

if USING_GPU:
    DEVICE = "cuda"
    INPUT_FOLDER = Path("/kaggle/input")
    IMAGE_SIZES = [256, 384, 512, 768, 1024]
    PHOTOS_PER_BATCH = 8
    PRETRAINED = True
    print("GPU:", torch.cuda.get_device_name(0))
    GPU_TOTAL_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("GPU memory in GB:", round(GPU_TOTAL_GB, 1))
else:
    # The laptop only proves the code runs: tiny photos and untrained networks
    DEVICE = "cpu"
    INPUT_FOLDER = Path("data/sample")
    IMAGE_SIZES = [64]
    PHOTOS_PER_BATCH = 2
    PRETRAINED = False
    GPU_TOTAL_GB = 0

VIAL_TRAINING_PHOTOS = 291
# anomalib's default for PatchCore: keep 10 percent of the memory bank
CORESET_RATIO = 0.1
# Reverse Distillation's optimizer settings, copied from anomalib's own configure_optimizers
RD_LEARNING_RATE = 0.005
RD_BETAS = (0.5, 0.99)
RESULTS_FILE = "gpu_size_results.csv"

vial_folder = None
for train_folder in INPUT_FOLDER.rglob("train"):
    if train_folder.parent.name == "vial":
        vial_folder = train_folder.parent
if vial_folder is None:
    raise FileNotFoundError("Could not find the Vial folder")

all_paths = sorted((vial_folder / "train" / "good").glob("*.png"))
# Evenly spread photos show more variety than the first few
step = len(all_paths) // PHOTOS_PER_BATCH
test_paths = all_paths[::step][:PHOTOS_PER_BATCH]
print("Photos used:", [path.name for path in test_paths])

with open(RESULTS_FILE, "w", newline="") as results_file:
    csv.writer(results_file).writerow(["size", "check", "measure", "value"])

## 2. Helper functions

- `prepare_batch` uses anomalib's default recipe at the chosen size, so the numbers match what a real run would do. It squashes to a square, as this check needs only the total number of pixels.
- `run_check` runs one check and records its results. If the GPU runs out of memory or reports another GPU error, it writes that down and carries on, because this is a useful answer. Each check is passed in as a function, so the same safety code wraps all of them. It also says how the check went. A GPU error jams the GPU for the rest of the session, so the main loop then stops, and everything measured so far stays saved. Running out of memory does not jam it, so the run carries on.
- `torch.cuda.max_memory_allocated` reports the most GPU memory used since the last reset. `torch.cuda.synchronize` waits for the GPU to finish, because otherwise the timer stops before the work is done.
- **Results are saved after each check,** so a problem in a later check does not lose earlier numbers.

In [ ]:
def record(size, check, measure, value):
    """Save one measured number straight away, and print it."""
    with open(RESULTS_FILE, "a", newline="") as results_file:
        csv.writer(results_file).writerow([size, check, measure, value])
    print(size, check, measure, value)


def prepare_batch(size):
    """Read the test photos and apply anomalib's default recipe at this size."""
    recipe = Padim.configure_pre_processor(image_size=(size, size)).transform
    photos = []
    for path in test_paths:
        photos.append(read_image(path, as_tensor=True))
    return recipe(torch.stack(photos)).to(DEVICE)


def start_timer():
    """Note the time, after waiting for the GPU to finish earlier work."""
    if USING_GPU:
        torch.cuda.synchronize()
    return time.perf_counter()


def stop_timer(started):
    """Return the seconds since the timer started."""
    if USING_GPU:
        torch.cuda.synchronize()
    return time.perf_counter() - started


def clear_memory():
    """Free memory, so one check cannot affect the next one's peak."""
    gc.collect()
    if USING_GPU:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def run_check(size, check_name, check_function):
    """Run one check, record its peak memory, and survive running out of memory."""
    clear_memory()
    try:
        check_function(size, check_name)
        if USING_GPU:
            record(size, check_name, "peak_gpu_gb", round(torch.cuda.max_memory_allocated() / 1e9, 2))
        outcome = "worked"
    except torch.cuda.OutOfMemoryError:
        outcome = "out of memory"
        record(size, check_name, "outcome", outcome)
    except torch.AcceleratorError as gpu_error:
        # Other GPU errors are recorded too, so one crash cannot lose the remaining checks
        outcome = "gpu error"
        record(size, check_name, "outcome", outcome + ": " + str(gpu_error).split(chr(10))[0])
        # Clearing memory would fail on a jammed GPU, so the run stops here
        return outcome
    clear_memory()
    return outcome

## 3. The checks

**PatchCore** (anomalib's defaults: `layers=["layer2", "layer3"]`, `num_neighbors=9`):
- `features`: in training mode, the model stores each photo's patch features in its memory bank.
- `coreset`: `subsample_embedding` keeps the most useful 10 percent. It runs on 8 photos here, so a full run takes far longer. Its work grows roughly with the square of the photo count, so the estimate is rough.
- `search`: a test photo is compared with a memory bank of the size a full run would keep. The bank holds random numbers, because only its size affects time and memory here. Its scores mean nothing and are not recorded.

**Reverse Distillation** (anomalib's defaults: `layers=["layer1", "layer2", "layer3"]`, `anomaly_map_mode=ADD`):
- Three real training steps. Training also works out corrections, so it uses more memory than just looking at photos. The first step includes set-up work, so it is left out of the time.

The number of Reverse Distillation training steps is not chosen yet, so only the time per step is given.

In [ ]:
def make_patchcore():
    """Build PatchCore's model with anomalib's default settings."""
    model = PatchcoreModel(
        backbone="wide_resnet50_2",
        pre_trained=PRETRAINED,
        layers=["layer2", "layer3"],
        num_neighbors=9,
    ).to(DEVICE)
    model.train()
    # The pretrained network must stay unchanged while features are collected
    model.feature_extractor.eval()
    return model


def check_patchcore_features(size, check_name):
    batch = prepare_batch(size)
    model = make_patchcore()
    model(batch)  # The first pass includes set-up work, so it is not timed
    started = start_timer()
    embedding = model(batch)
    seconds = stop_timer(started)

    patches_per_photo = embedding.shape[0] // PHOTOS_PER_BATCH
    feature_length = embedding.shape[1]
    # 4 bytes per number, because the model uses 32-bit decimal numbers
    full_bank_gb = VIAL_TRAINING_PHOTOS * patches_per_photo * feature_length * 4 / 1e9
    record(size, check_name, "seconds_per_photo", round(seconds / PHOTOS_PER_BATCH, 4))
    record(size, check_name, "patches_per_photo", patches_per_photo)
    record(size, check_name, "feature_length", feature_length)
    record(size, check_name, "estimated_full_bank_before_sampling_gb", round(full_bank_gb, 2))


def check_patchcore_coreset(size, check_name):
    batch = prepare_batch(size)
    model = make_patchcore()
    model(batch)
    started = start_timer()
    model.subsample_embedding(CORESET_RATIO)
    seconds = stop_timer(started)
    scale = VIAL_TRAINING_PHOTOS / PHOTOS_PER_BATCH
    record(size, check_name, "seconds_on_test_batch", round(seconds, 2))
    record(size, check_name, "rough_estimate_full_run_minutes", round(seconds * scale * scale / 60, 1))


def check_patchcore_search(size, check_name):
    batch = prepare_batch(size)
    model = make_patchcore()
    embedding = model(batch)
    patches_per_photo = embedding.shape[0] // PHOTOS_PER_BATCH
    bank_rows = int(VIAL_TRAINING_PHOTOS * patches_per_photo * CORESET_RATIO)
    # Only the bank's size matters for time and memory, so its numbers can be random
    model.memory_bank = torch.randn(bank_rows, embedding.shape[1], device=DEVICE)
    model.embedding_store.clear()
    model.eval()
    model(batch)
    started = start_timer()
    model(batch)
    seconds = stop_timer(started)
    record(size, check_name, "seconds_per_photo", round(seconds / PHOTOS_PER_BATCH, 4))
    record(size, check_name, "memory_bank_gb", round(bank_rows * embedding.shape[1] * 4 / 1e9, 2))


def check_reverse_distillation(size, check_name):
    batch = prepare_batch(size)
    model = ReverseDistillationModel(
        backbone="wide_resnet50_2",
        input_size=(size, size),
        layers=["layer1", "layer2", "layer3"],
        anomaly_map_mode=AnomalyMapGenerationMode.ADD,
        pre_trained=PRETRAINED,
    ).to(DEVICE)
    model.train()
    loss_function = ReverseDistillationLoss()
    # As in anomalib, only the decoder and bottleneck learn. The encoder stays fixed
    trainable = list(model.decoder.parameters()) + list(model.bottleneck.parameters())
    optimizer = torch.optim.Adam(params=trainable, lr=RD_LEARNING_RATE, betas=RD_BETAS)

    step_seconds = []
    for step_number in range(3):
        started = start_timer()
        optimizer.zero_grad()
        loss = loss_function(*model(batch))
        loss.backward()
        optimizer.step()
        step_seconds.append(stop_timer(started))
    # The first step includes set-up work, so it is left out
    record(size, check_name, "seconds_per_training_step", round(sum(step_seconds[1:]) / 2, 3))


CHECKS = [
    ("patchcore_features", check_patchcore_features),
    ("patchcore_coreset", check_patchcore_coreset),
    ("patchcore_search", check_patchcore_search),
    ("reverse_distillation", check_reverse_distillation),
]

whole_run_started = time.perf_counter()
stopped = False
for size in IMAGE_SIZES:
    for check_name, check_function in CHECKS:
        outcome = run_check(size, check_name, check_function)
        if outcome == "gpu error":
            stopped = True
            break
    if stopped:
        break
print("Whole run, minutes:", round((time.perf_counter() - whole_run_started) / 60, 1))